In [1]:
import numpy as np
import torch
import hydra
from hydra import initialize, compose
from omegaconf import DictConfig

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict

from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

In [2]:
from os.path import join, dirname
import os
import sys

from einops import rearrange, repeat

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from memoryPerceiver.memory_perceiver.main.configuration import cf

from memoryPerceiver.memory_perceiver.core.model_net import PerceiverIO
from memoryPerceiver.memory_perceiver.processing.vectorize import input_to_tensor, fourier_encode

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

env: ISAAC_JUPYTER_KERNEL=1


In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
nb_frames = 32
frames_future = 8
len_video_frame = 1824  # 456*256/(8*8)
len_audio_frame = 120   # 1920/16
query_dim = 832
video_latent_dim = 256
audio_latent_dim = 560
num_workers = 2
batch_size = 4

len_vid = nb_frames*len_video_frame
len_audio = nb_frames*len_audio_frame
len_audio_raw = len_audio*cf.sample_per_frame
video_future = frames_future*len_video_frame
audio_future = frames_future*len_audio_frame

special_compression = [i for i in range(video_future)] + [i for i in range(len_vid, len_vid+audio_future)]
for k in range(len(cf.body_parts)):
    special_compression += [ len_vid + len_audio + k*nb_frames + i for i in range(frames_future) ]

In [5]:
def create_model(device=device):

    # Projection to return modalities, enhancement

    modality_parameters = {}
    modality_parameters['audio'] = audio_latent_dim
    modality_parameters['video'] = video_latent_dim
    for part, params in cf.body_parts.items():
        modality_parameters[part] = query_dim - 2*128 - params
    for part, params in cf.body_outputs.items():
        modality_parameters[part] = query_dim - 2*128 - params

    projection_parameters = dict(cf.body_outputs)
    projection_parameters.update({'audio': 16})

    # Instantiate model
        
    model = PerceiverIO(
            2,
            query_dim,
            special_compression = special_compression,
            device=device,
            projection_parameters = projection_parameters,
            modality_parameters = modality_parameters,
            max_batch_size = batch_size
        )
    model = model.to(device)

    # Queries

    queries = []
    audio_query = repeat( torch.cat([model.modality['audio'], torch.zeros(16, device=device)]), '... -> n ...', n=audio_future)
    audio_pos = fourier_encode(torch.linspace(-1, 1, audio_future), audio_future/2, num_bands=128).to(device)
    queries.append(torch.cat([audio_query, audio_pos], dim=-1).to(device))
    for part, params in cf.body_outputs.items():
        part_query = repeat( torch.cat([model.modality[part], torch.zeros(params, device=device)]), '... -> n ...', n=frames_future)
        part_pos = fourier_encode(torch.linspace(-1, 1, frames_future), frames_future/2, num_bands=128).to(device)
        queries.append(torch.cat([part_query, part_pos], dim=-1).to(device))
    queries=torch.cat(queries)


    return model, queries

In [6]:
model, queries = create_model()

inputs={}
inputs['video']=torch.randn(batch_size, 3, nb_frames, 256, 456)
inputs['audio']=torch.randn(batch_size, nb_frames*1920)
inputs['observations']=torch.randn(batch_size, nb_frames, 4)

batch = input_to_tensor(inputs, model.modality, cf.body_parts)

In [7]:
with torch.no_grad():
    x, aux_loss = model(batch.to(device), queries = queries)
    output = {}
    output['audio'] = rearrange(model.projections['audio'](x[:, :audio_future, :]) ,  "b t t1 -> b (t t1)")
    start_modality = audio_future
    for part, params in cf.body_outputs.items():
        output[part] = model.projections[part](x[:, start_modality:start_modality+frames_future, :])
        start_modality += frames_future

In [8]:
rearrange(output['actions'], 'b t d -> t b d')

tensor([[[0.0109],
         [0.0106],
         [0.0108],
         [0.0107]],

        [[0.0091],
         [0.0088],
         [0.0090],
         [0.0090]],

        [[0.0094],
         [0.0091],
         [0.0094],
         [0.0093]],

        [[0.0118],
         [0.0115],
         [0.0117],
         [0.0116]],

        [[0.0121],
         [0.0117],
         [0.0120],
         [0.0119]],

        [[0.0098],
         [0.0095],
         [0.0098],
         [0.0097]],

        [[0.0095],
         [0.0092],
         [0.0094],
         [0.0093]],

        [[0.0103],
         [0.0100],
         [0.0102],
         [0.0101]]], device='cuda:0')

In [9]:
output['actions']

tensor([[[0.0109],
         [0.0091],
         [0.0094],
         [0.0118],
         [0.0121],
         [0.0098],
         [0.0095],
         [0.0103]],

        [[0.0106],
         [0.0088],
         [0.0091],
         [0.0115],
         [0.0117],
         [0.0095],
         [0.0092],
         [0.0100]],

        [[0.0108],
         [0.0090],
         [0.0094],
         [0.0117],
         [0.0120],
         [0.0098],
         [0.0094],
         [0.0102]],

        [[0.0107],
         [0.0090],
         [0.0093],
         [0.0116],
         [0.0119],
         [0.0097],
         [0.0093],
         [0.0101]]], device='cuda:0')

In [10]:
stack = []
for i in range(32):
    stack.append(torch.randn(8,4))
(rearrange(stack, 't b d -> b t d')).size()

torch.Size([8, 32, 4])

In [11]:
list(torch.unbind(torch.randn(8,4)))

[tensor([-0.7508,  2.3570, -0.3946,  0.4513]),
 tensor([ 0.1984,  0.5892,  0.3256, -0.6042]),
 tensor([-0.3356, -0.3171, -0.0238, -0.8681]),
 tensor([-0.6712,  0.2723,  0.2534,  0.2474]),
 tensor([ 2.3932,  0.1460, -0.2401, -0.8180]),
 tensor([ 0.2664,  0.4532,  0.6491, -0.6373]),
 tensor([-1.2077,  0.2508,  0.5496, -0.1825]),
 tensor([-0.3572, -0.5169, -0.1048, -0.6656])]

In [12]:
history_keys = ['video', 'audio', 'obs', 'reward']
{key:[] for key in history_keys}.update({'dfsfs':[]})

In [13]:
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def parse_hydra_configs(cfg: DictConfig):

    cfg_dict = omegaconf_to_dict(cfg)
    print_dict(cfg_dict)

    headless = cfg.headless
    render = not headless

    env = VecEnvRLGames(headless=headless, sim_device=cfg.device_id)
    # sets seed. if seed is -1 will pick a random one
    from omni.isaac.core.utils.torch.maths import set_seed
    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed
    task = initialize_task(cfg_dict, env)

    history = {'video':[], 'audio':[], 'obs':[], 'reward':[], 'action':[]}
    buffer = {'video':[], 'audio':[], 'obs':[], 'reward':[], 'action':[]}
    while env._simulation_app.is_running():
        if env._world.is_playing():
            if env._world.current_time_step_index == 0:
                env._world.reset(soft=True)

            # Observations: 'cartpole_view' 'obs_buf' (numEnvs, 4)

            obs = env._task.get_observations()['cartpole_view']['obs_buf']
            buffer['obs'].append(obs)
            history['obs'].append(obs)
            if(len(buffer['obs'])>nb_frames):
                buffer['obs'].pop(0)

            # Reward (numEnvs, 1)

            reward=env._task.calculate_metrics()
            buffer['reward'].append(obs)
            history['reward'].append(reward)
            if(len(buffer['reward'])>nb_frames):
                 buffer['reward'].pop(0)

            # Process input

            inputs={}
            inputs['video']=torch.randn(numEnvs, 3, nb_frames, 256, 456)
            inputs['audio']=torch.randn(numEnvs, nb_frames*1920)
            inputs['observations']=rearrange(buffer['obs'], 't b d -> b t d')
            batch = input_to_tensor(inputs, model.modality, cf.body_parts)

            # Agent action

            output = {}
            if(len(buffer['action']) == 0 and len(buffer['obs']) >= nb_frames):
                with torch.no_grad():
                    x, aux_loss = model(batch.to(device), queries = queries)
                    output['audio'] = rearrange(model.projections['audio'](
                        x[:, :audio_future, :]), "b t t1 -> b (t t1)"
                    )
                    start_modality = audio_future
                    for part, params in cf.body_outputs.items():
                        output[part] = model.projections[part](
                            x[:, start_modality:start_modality+frames_future, :]
                        )
                        start_modality += frames_future
                print('ACTIONS *****', output['actions'])
                new_actions = list(torch.unbind(
                                    rearrange(output['actions'], 'b t d -> t b d')
                                ))
                buffer['action'] += new_actions
                history['action'] += new_actions

            # Save video, audio, body, reward
            if env.sim_frame_count % save_step == 0 and len(history['obs']) >= 0:
                print('obs', buffer['obs'], 'reward', buffer['reward'], 'action', buffer['action'])
                print("$$$$$$$$$$$$ 0 &&& 0 $$$$$$$$$$$")
                for key in history.keys():
                    history[key]=[]

            # Current action (numEnvs, 1)
            # if(len(action_buffer)>0):
            #     action = action_buffer.pop(0)
            # else:
            #     action = torch.randn(numEnvs, 1)

            # Step environment

            action = torch.tensor(np.array([env.action_space.sample() for _ in range(env.num_envs)]), device=task.rl_device)
            env._task.pre_physics_step(action.to(task.rl_device))
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()
        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    save_step = 160
    numEnvs = 4
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Cartpole", "headless=True","+numEnvs="+str(numEnvs)])
    parse_hydra_configs(cfg)

/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


task: 
    name: Cartpole
    physics_engine: physx
    env: 
        numEnvs: 4
        envSpacing: 4.0
        resetDist: 3.0
        maxEffort: 400.0
        clipObservations: 5.0
        clipActions: 1.0
        controlFrequencyInv: 2
    sim: 
        dt: 0.0083
        use_gpu_pipeline: True
        gravity: [0.0, 0.0, -9.81]
        add_ground_plane: True
        add_distant_light: True
        use_flatcache: True
        enable_scene_query_support: False
        disable_contact_processing: False
        enable_cameras: False
        default_physics_material: 
            static_friction: 1.0
            dynamic_friction: 1.0
            restitution: 0.0
        physx: 
            worker_thread_count: 4
            solver_type: 1
            use_gpu: True
            solver_position_iteration_count: 4
            solver_velocity_iteration_count: 0
            contact_offset: 0.02
            rest_offset: 0.001
            bounce_threshold_velocity: 0.2
            friction_offs

/isaac-sim/exts/omni.isaac.ml_archive/pip_prebundle/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


[9.926s] [ext: omni.kit.viewport.utility-1.0.12] shutdown
[9.926s] [ext: omni.kit.browser.sample-1.2.4] shutdown
[9.927s] [ext: omni.kit.browser.folder.core-1.7.1] shutdown
[9.927s] [ext: omni.kit.browser.core-2.2.1] shutdown
[9.928s] [ext: omni.kit.viewport.window-104.1.18] shutdown
2023-02-22 10:35:32 [9,947ms] [Warning] [omni.ext._impl._internal] omni.kit.viewport.window-104.1.18 -> <class 'omni.kit.viewport.window.extension.ViewportWindowExtension'>: extension object is still alive, something holds a reference on it. References: ["[0]:type: <class 'frame'>, id: 139713010123456", "[1]:type: <class 'frame'>, id: 139713009974256", "[2]:type: <class 'frame'>, id: 1653896848"]
2023-02-22 10:35:32 [9,964ms] [Warning] [omni.isaac.core.utils.nucleus] /persistent/app/omniverse/mountedDrives setting not found
2023-02-22 10:35:33 [10,897ms] [Warning] [omni.isaac.core.utils.nucleus] /persistent/app/omniverse/mountedDrives setting not found
obs [tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.

KeyError: 'action'

In [ ]:
%tb

No traceback available to show.
